In [1]:
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, logging, RobertaModel
import torch

from dataset import CL_Dataset, CustomSubset
from model import SiameseNetwork, CustomContrastiveLoss

logging.set_verbosity_error()

c:\Users\Sergio\miniconda3\envs\mbib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Parameters
#Data

train_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
val_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/val']
mention_type = "event"
context_method = "hsu"
train_dataset_name = "ecbplus_train"
val_dataset_name = "ecbplus_val"
language_model = 'roberta-base'
special_tokens = ('[E]', '[/E]')

#Model
gpu_num = [0]
margin=0.4

#Optimizer and learning
learning_rate = 2e-7
weight_decay = 1e-2
num_epochs = 100
train_batch_size = 4
val_batch_size = 32

In [3]:
#Initialize model and tokenizer
roberta = RobertaModel.from_pretrained(pretrained_model_name_or_path= language_model) 
tokenizer = RobertaTokenizer.from_pretrained(language_model)
tokenizer.add_tokens(special_tokens)

2

In [30]:
#Initialization
#Load data

cl_cdcr_train = CL_Dataset(name=train_dataset_name,data_paths=train_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="train") 

cl_cdcr_val = CL_Dataset(name=val_dataset_name,data_paths=val_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="inference") 

#Use subset for testing:
#cl_cdcr_val = CustomSubset(cl_cdcr_val,[0,1,2,3])

#Keep in mind ECB topic repeated in train and topic. topic 34 I think

train_dataloader= DataLoader(cl_cdcr_train, batch_size=train_batch_size, shuffle=True)

val_dataloader= DataLoader(cl_cdcr_val, batch_size=val_batch_size, shuffle=False)

In [40]:
31*32

992

In [37]:
len(val_dataloader)

31

In [31]:
len(cl_cdcr_val.keys_list)

986

In [20]:
# Initialize model
device_ids = gpu_num
device = torch.device("cuda:{}".format(device_ids[0]))
#Temporary use of cpu
device = torch.device("cpu")

siamese_model_single = SiameseNetwork(model=roberta,tokenizer=tokenizer).to(device)
model=siamese_model_single
# To paralelize
#siamese_model = torch.nn.DataParallel(siamese_model_single, device_ids=device_ids)
#model = siamese_model_single


#

In [21]:
# Initialize optimizer
#Parameters
# Define the learning rate and weight decay

# Define optimizer!
# AdamW optimizer
# Epochs: 100 Events 50 Entities
# Batch size 16
# m 0.4 0.7
# Clistering Threshold
# Learning rate  2e-7 2e-7


optimizer_params = [
    {'params': model.language_model.parameters(), 'lr': learning_rate},
    {'params': model.mlp_cls_embed.parameters(), 'lr': learning_rate},
    {'params': model.mlp_mention_embed.parameters(), 'lr': learning_rate},
]

contrastive_loss = CustomContrastiveLoss(margin=margin)
optimizer = torch.optim.AdamW(optimizer_params)

In [41]:
torch.zeros(32)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.])

In [49]:
#Store vector representations by topic
dict_embeddings_by_topic={}
#Validation dataloader
for indices, array_mention in val_dataloader: #Validation batch size should be different. It will be way faster
    # I also need to get the keys from the mentions to cluster them later
    with torch.no_grad():
        output_mentions=model(array_mention)
        batch_keys = [cl_cdcr_val.keys_list[i] for i in indices]

        for key, embedding in zip(batch_keys,output_mentions):
            mention_temp = cl_cdcr_val.cdcr_dataset.mentions[key]
            topic_number = mention_temp.topic_id

            #Stores embeddings by topic
            if topic_number in dict_embeddings_by_topic:
                dict_embeddings_by_topic[topic_number][key] = embedding
            else:
                dict_embeddings_by_topic[topic_number] = {key: embedding}

In [50]:
dict_embeddings_by_topic["12"]

{'12_10ecb_ACT17403639225065902_0_4_F4LF': tensor([ 0.0669, -0.0355, -0.0135,  ..., -0.3405,  0.0910, -0.2736]),
 '12_11ecb_ACT17403639225065902_0_5_TERD': tensor([ 0.0649, -0.0348, -0.0168,  ..., -0.3287,  0.1412, -0.3529]),
 '12_12ecb_ACT17403639225065902_0_23_VvP8': tensor([ 0.0677, -0.0353, -0.0170,  ..., -0.3870,  0.1121, -0.4660]),
 '12_13ecb_ACT17403639225065902_0_3_i5A7': tensor([ 0.0655, -0.0356, -0.0159,  ..., -0.3304,  0.0867, -0.3136]),
 '12_14ecb_ACT17403639225065902_0_4_DtDL': tensor([ 0.0648, -0.0369, -0.0158,  ..., -0.3383,  0.0883, -0.2870]),
 '12_15ecb_ACT17403639225065902_0_10_kfy2': tensor([ 0.0647, -0.0343, -0.0131,  ..., -0.3438,  0.0996, -0.2665]),
 '12_16ecb_ACT17403639225065902_0_17_6LUA': tensor([ 0.0656, -0.0342, -0.0151,  ..., -0.3037,  0.1517, -0.2926]),
 '12_17ecb_ACT17403639225065902_0_23_BqbC': tensor([ 0.0663, -0.0367, -0.0161,  ..., -0.5831,  0.3382, -0.9394]),
 '12_18ecb_ACT17403639225065902_0_7_TyQk': tensor([ 0.0652, -0.0365, -0.0154,  ..., -0.3671,

In [ ]:
#Training loop
for epoch in range(num_epochs):
    accumulate_loss = 0
    model.train()

    #Train dataloader
    for array_mention1,array_mention2, labels in train_dataloader:


        optimizer.zero_grad()  # Zero the gradients
        output_mentions1 = model(array_mention1)
        output_mentions2 = model(array_mention2)
        
        loss = contrastive_loss(output_mentions1, output_mentions2, labels)

        loss.backward()  # Backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()  # Optimization step

        torch.cuda.empty_cache()
        accumulate_loss += loss.item()

    #Validation
    model.eval()
    #Store vector representations by topic
    dict_embeddings_by_topic={}
    #Validation dataloader
    for indices, array_mention in val_dataloader: #Validation batch size should be different. It will be way faster
        # I also need to get the keys from the mentions to cluster them later
        with torch.no_grad():
            output_mentions=model(array_mention)

            batch_keys = [cl_cdcr_val.keys_list[i] for i in indices]

            for key, embedding in zip(batch_keys,output_mentions):
                mention_temp = cl_cdcr_val.cdcr_dataset.mentions[key]
                topic_number = mention_temp.topic_id

                #Stores embeddings by topic
                if topic_number in dict_embeddings_by_topic:
                    dict_embeddings_by_topic[topic_number][key] = embedding
                else:
                    dict_embeddings_by_topic[topic_number] = {key: embedding}
        
        #Once I store all the values by topic, I run the agglomerative clustering by topic and compute F1-CoNLL score
            



    # Optionally, you can print or log the loss for monitoring training progress
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')